In [16]:
# ============================================================
# Inicialização do ambiente - Camada Trusted/Silver
# Objetivo:
#   Importar bibliotecas necessárias e carregar os datasets
#   tratados a partir da camada trusted (silver).
# ============================================================

# ------------------------------------------------------------
# 1) Importação de bibliotecas
# ------------------------------------------------------------
# pandas: manipulação e análise de dados
# numpy: operações numéricas vetorizadas
# pathlib.Path: manipulação segura de caminhos

import pandas as pd
import numpy as np
from pathlib import Path


# ------------------------------------------------------------
# 2) Definição do caminho da camada Trusted/Silver
# ------------------------------------------------------------
# Boa prática:
# - Centralizar caminho base da camada
# - Facilitar manutenção e portabilidade do projeto

TRUSTED_PATH = Path("../data/trusted_silver")


# ------------------------------------------------------------
# 3) Leitura dos arquivos Parquet
# ------------------------------------------------------------
# Boa prática:
# - Utilizar Parquet por eficiência e preservação de schema
# - Confirmar que tipos como datetime e Int64 foram mantidos

df_bairros = pd.read_parquet(TRUSTED_PATH / "bairros.parquet")
df_conc = pd.read_parquet(TRUSTED_PATH / "concorrentes.parquet")
df_eventos = pd.read_parquet(TRUSTED_PATH / "eventos.parquet")
df_pop = pd.read_parquet(TRUSTED_PATH / "populacao.parquet")


ENRIQUECIMENTO TEMPORAL (EVENTOS)

In [17]:
# ============================================================
# Enriquecimento temporal - df_eventos
# Objetivo:
#   Garantir tipagem correta da coluna datetime e criar
#   variáveis derivadas para análise temporal.
# ============================================================

# ------------------------------------------------------------
# 1) Garantir tipagem datetime
# ------------------------------------------------------------
# Boa prática:
# - Reforçar tipagem após leitura
# - Evitar erros ao acessar atributos .dt

df_eventos["datetime"] = pd.to_datetime(df_eventos["datetime"])


# ------------------------------------------------------------
# 2) Extração da data (sem horário)
# ------------------------------------------------------------
# Cria coluna apenas com a parte da data.
# Útil para agregações diárias.

df_eventos["data"] = df_eventos["datetime"].dt.date


# ------------------------------------------------------------
# 3) Extração do dia da semana
# ------------------------------------------------------------
# day_name() retorna o nome do dia em inglês.
# Caso necessário, pode-se configurar locale para português.

df_eventos["dia_semana"] = df_eventos["datetime"].dt.day_name()


# ------------------------------------------------------------
# 4) Extração da hora
# ------------------------------------------------------------
# Útil para análises por faixa horária.

df_eventos["hora"] = df_eventos["datetime"].dt.hour


# ------------------------------------------------------------
# 5) Classificação por período do dia
# ------------------------------------------------------------
# Boa prática:
# - Definir regra clara de negócio
# - Centralizar lógica de classificação em função separada

def classificar_periodo(hora):
    """
    Classifica hora do dia em períodos:
        - manha: 05h às 11h59
        - tarde: 12h às 17h59
        - noite: 18h às 04h59
    """
    if 5 <= hora < 12:
        return "manha"
    elif 12 <= hora < 18:
        return "tarde"
    else:
        return "noite"


# Aplicação da classificação
df_eventos["periodo"] = df_eventos["hora"].apply(classificar_periodo)
df_eventos

,codigo,datetime,codigo_concorrente,data,dia_semana,hora,periodo
0,oMn07h1bJYV0Wdx+RTzsDcT8JQlT7QXc7q8A/4y+cO5gBQ...,2017-07-27 09:51:02.000,650509405109544,2017-07-27,Thursday,9,manha
1,iZuQeTd9am+qfaiqnn5kkixogIbwN0nY2gtMwZqH9bFqph...,2017-06-24 14:00:26.405,650509405109544,2017-06-24,Saturday,14,tarde
2,iIMvwWSnQoW0aqQlcvjc8A6LvjkRX1HLppdkdQZapPVVv7...,2017-07-06 21:51:11.056,650509405109544,2017-07-06,Thursday,21,noite
3,jYaJZuli3fVSNDOs60if9VObBmCOoJP2x9H3IZP+6FsMRk...,2017-07-02 14:27:09.316,650509405109544,2017-07-02,Sunday,14,tarde
4,jYaJZuli3fVSNDOs60if9VObBmCOoJP2x9H3IZP+6FsMRk...,2017-07-15 03:03:29.731,650509405109544,2017-07-15,Saturday,3,noite
...,...,...,...,...,...,...,...
209796,jYaJZuli3fVSNDOs60if9VObBmCOoJP2x9H3IZP+6FsMRk...,2017-07-02 05:53:55.683,650509405109544,2017-07-02,Sunday,5,manha
209797,L2LNsLdlJMdr6HeuMICn51YlaE2rD6OhOXcXbsjjqG2yCd...,2017-07-19 13:09:35.574,650509405109544,2017-07-19,Wednesday,13,tarde
209798,jYaJZuli3fVSNDOs60if9VObBmCOoJP2x9H3IZP+6FsMRk...,2017-07-14 21:01:34.724,650509405109544,2017-07-14,Friday,21,noite
209799,JU6Ggudehig6AffZFq+8xDe/MttO8m5P5+s+ANmV+JkGnv...,2017-07-04 23:48:31.197,650509405109544,2017-07-04,Tuesday,23,noite


In [18]:
# ============================================================
# Enriquecimento temporal - df_eventos
# Objetivo:
#   Garantir tipagem correta da coluna datetime e criar
#   atributos derivados para análises por data e período.
# ============================================================

# ------------------------------------------------------------
# 1) Garantir tipagem datetime
# ------------------------------------------------------------
# Boa prática:
# - Reforçar tipagem após leitura de arquivos
# - Evitar erro ao utilizar o accessor .dt

df_eventos["datetime"] = pd.to_datetime(df_eventos["datetime"])


# ------------------------------------------------------------
# 2) Criar coluna apenas com a data
# ------------------------------------------------------------
# Remove a parte de hora/minuto/segundo.
# Útil para agregações diárias.

df_eventos["data"] = df_eventos["datetime"].dt.date


# ------------------------------------------------------------
# 3) Criar coluna com dia da semana
# ------------------------------------------------------------
# day_name() retorna o nome do dia em inglês.
# Caso necessário, pode-se traduzir posteriormente.

df_eventos["dia_semana"] = df_eventos["datetime"].dt.day_name()


# ------------------------------------------------------------
# 4) Criar coluna com hora
# ------------------------------------------------------------
# Permite análises por faixa horária.

df_eventos["hora"] = df_eventos["datetime"].dt.hour


# ------------------------------------------------------------
# 5) Classificação por período do dia
# ------------------------------------------------------------
# Boa prática:
# - Centralizar regra de negócio em função específica
# - Documentar claramente os intervalos

def classificar_periodo(hora):
    """
    Classificação do período:
        - manha: 05h até 11h
        - tarde: 12h até 17h
        - noite: 18h até 04h
    """
    if 5 <= hora < 12:
        return "manha"
    elif 12 <= hora < 18:
        return "tarde"
    else:
        return "noite"


# Aplicação da regra de classificação
# Observação:
# - .apply() é simples e legível
# - Para grandes volumes, considerar abordagem vetorizada

df_eventos["periodo"] = df_eventos["hora"].apply(classificar_periodo)
df_eventos

,codigo,datetime,codigo_concorrente,data,dia_semana,hora,periodo
0,oMn07h1bJYV0Wdx+RTzsDcT8JQlT7QXc7q8A/4y+cO5gBQ...,2017-07-27 09:51:02.000,650509405109544,2017-07-27,Thursday,9,manha
1,iZuQeTd9am+qfaiqnn5kkixogIbwN0nY2gtMwZqH9bFqph...,2017-06-24 14:00:26.405,650509405109544,2017-06-24,Saturday,14,tarde
2,iIMvwWSnQoW0aqQlcvjc8A6LvjkRX1HLppdkdQZapPVVv7...,2017-07-06 21:51:11.056,650509405109544,2017-07-06,Thursday,21,noite
3,jYaJZuli3fVSNDOs60if9VObBmCOoJP2x9H3IZP+6FsMRk...,2017-07-02 14:27:09.316,650509405109544,2017-07-02,Sunday,14,tarde
4,jYaJZuli3fVSNDOs60if9VObBmCOoJP2x9H3IZP+6FsMRk...,2017-07-15 03:03:29.731,650509405109544,2017-07-15,Saturday,3,noite
...,...,...,...,...,...,...,...
209796,jYaJZuli3fVSNDOs60if9VObBmCOoJP2x9H3IZP+6FsMRk...,2017-07-02 05:53:55.683,650509405109544,2017-07-02,Sunday,5,manha
209797,L2LNsLdlJMdr6HeuMICn51YlaE2rD6OhOXcXbsjjqG2yCd...,2017-07-19 13:09:35.574,650509405109544,2017-07-19,Wednesday,13,tarde
209798,jYaJZuli3fVSNDOs60if9VObBmCOoJP2x9H3IZP+6FsMRk...,2017-07-14 21:01:34.724,650509405109544,2017-07-14,Friday,21,noite
209799,JU6Ggudehig6AffZFq+8xDe/MttO8m5P5+s+ANmV+JkGnv...,2017-07-04 23:48:31.197,650509405109544,2017-07-04,Tuesday,23,noite


FLUXO DIÁRIO POR CONCORRENTE

In [19]:
# ============================================================
# Cálculo de fluxo diário por concorrente
# Objetivo:
#   Agregar eventos para obter o total de ocorrências por:
#       - codigo_concorrente
#       - data
#       - dia_semana
#       - periodo
# ============================================================

# ------------------------------------------------------------
# 1) Agrupamento dos eventos
# ------------------------------------------------------------
# groupby:
# - Define a granularidade da agregação
# - Neste caso, o nível é diário por concorrente e período
#
# .size():
# - Conta o número de registros em cada grupo
# - Equivalente a COUNT(*) em SQL
#
# Boa prática:
# - Garantir que a coluna "data" esteja corretamente derivada
# - Garantir que não existam duplicidades antes da agregação

fluxo_diario = (
    df_eventos
    .groupby(["codigo_concorrente", "data", "dia_semana", "periodo"])
    .size()
    .reset_index(name="total_eventos_dia")
)


# ------------------------------------------------------------
# 2) Resultado
# ------------------------------------------------------------
# reset_index(name="total_eventos_dia"):
# - Converte o resultado em DataFrame estruturado
# - Nomeia explicitamente a métrica agregada
#
# Boa prática:
# - Nomear métricas de forma descritiva
# - Facilitar entendimento do indicador

fluxo_diario

,codigo_concorrente,data,dia_semana,periodo,total_eventos_dia
0,101025009987066,2017-06-26,Monday,noite,1
1,101025009987066,2017-06-30,Friday,noite,3
2,101025009987066,2017-07-01,Saturday,manha,4
3,101025009987066,2017-07-01,Saturday,noite,11
4,101025009987066,2017-07-01,Saturday,tarde,4
...,...,...,...,...,...
40277,1897386380476876,2017-07-27,Thursday,manha,1
40278,1897386380476876,2017-07-28,Friday,noite,1
40279,1897386380476876,2017-07-29,Saturday,manha,2
40280,1897386380476876,2017-07-29,Saturday,noite,1


MÉTRICAS DE FLUXO (MÉDIA, MAX, MIN)

In [20]:
# ============================================================
# Construção de métricas analíticas de fluxo
# Objetivo:
#   Gerar indicadores agregados por:
#       - codigo_concorrente
#       - dia_semana
#       - periodo
#
#   Métricas calculadas:
#       - média de eventos diários
#       - máximo diário
#       - mínimo diário
#       - total acumulado de eventos
# ============================================================

# ------------------------------------------------------------
# 1) Agrupamento analítico
# ------------------------------------------------------------
# A granularidade agora deixa de ser diária e passa a ser:
#   concorrente + dia da semana + período
#
# Isso permite identificar padrões médios de comportamento.

fluxo_analitico = (
    fluxo_diario
    .groupby(["codigo_concorrente", "dia_semana", "periodo"])
    
    # --------------------------------------------------------
    # 2) Definição das métricas
    # --------------------------------------------------------
    # .agg() com named aggregation:
    # - Permite nomear explicitamente cada métrica
    # - Melhora legibilidade e organização
    #
    # Boa prática:
    # - Usar nomes claros e padronizados
    # - Evitar abreviações ambíguas
    
    .agg(
        media_fluxo=("total_eventos_dia", "mean"),
        max_fluxo=("total_eventos_dia", "max"),
        min_fluxo=("total_eventos_dia", "min"),
        total_eventos=("total_eventos_dia", "sum")
    )
    
    # --------------------------------------------------------
    # 3) Retornar como DataFrame estruturado
    # --------------------------------------------------------
    # reset_index() remove índice hierárquico
    
    .reset_index()
)

fluxo_analitico

,codigo_concorrente,dia_semana,periodo,media_fluxo,max_fluxo,min_fluxo,total_eventos
0,101025009987066,Friday,manha,4.000000,6,3,16
1,101025009987066,Friday,noite,2.200000,3,1,11
2,101025009987066,Friday,tarde,5.250000,9,3,21
3,101025009987066,Monday,manha,3.600000,5,2,18
4,101025009987066,Monday,noite,2.600000,4,1,13
...,...,...,...,...,...,...,...
12267,1897386380476876,Thursday,tarde,1.333333,2,1,4
12268,1897386380476876,Tuesday,noite,1.000000,1,1,1
12269,1897386380476876,Tuesday,tarde,1.000000,1,1,3
12270,1897386380476876,Wednesday,noite,1.000000,1,1,2


FAIXA DE PREÇO DOS CONCORRENTES

In [21]:
# ============================================================
# Preparação da dimensão de concorrentes (faixa de preço)
# Objetivo:
#   Selecionar colunas relevantes da base de concorrentes
#   e padronizar o nome da chave para integração futura.
# ============================================================

# ------------------------------------------------------------
# 1) Seleção de colunas relevantes
# ------------------------------------------------------------
# Mantemos apenas:
# - codigo (identificador do concorrente)
# - nome
# - categoria
# - faixa_preco
#
# Boa prática:
# - Reduzir dataset ao necessário para análise
# - Evitar carregar colunas desnecessárias em joins

faixa_preco = df_conc[[
    "codigo",
    "nome",
    "categoria",
    "faixa_preco"
]]


# ------------------------------------------------------------
# 2) Padronização da chave
# ------------------------------------------------------------
# Renomeamos "codigo" para "codigo_concorrente"
# para manter consistência com a base de eventos.
#
# Boa prática:
# - Padronizar nomes de chave entre datasets
# - Facilitar joins e evitar ambiguidades

faixa_preco = faixa_preco.rename(
    columns={"codigo": "codigo_concorrente"}
)


# ------------------------------------------------------------
# 3) Resultado
# ------------------------------------------------------------
# Dataset pronto para join com fluxo_analitico
# ou outras tabelas de eventos.

faixa_preco

,codigo_concorrente,nome,categoria,faixa_preco
0,431962533652067,Boizão Lanches,Bar,2
1,1663855903830869,Bar do Serjão,Bar,0
2,567824576564110,Recanto Do Kuca,Restaurant,0
3,202740866540615,Dedé Abelhuda,Grocery Store,0
4,1784900838394305,Tenshi Sushi Boteco Itu,Sushi Restaurant,3
...,...,...,...,...
4197,1831179237099774,Supermercado Santana,"Grocery Store, Food & Beverage Company, Market",0
4198,902572199806965,M&S bolos e doces.,Grocery Store,0
4199,1594553520828717,Nai Cakes,"Cupcake Shop, Tea Room",0
4200,1754208714805968,Flamel,"Dessert Shop, Grocery Store",0


GEOGRAFIA + DENSIDADE DEMOGRÁFICA

In [22]:
# ============================================================
# Integração Geográfica e Cálculo de Densidade Demográfica
# Objetivo:
#   Integrar dados de concorrentes, bairros e população,
#   garantindo integridade de chaves e calculando densidade
#   demográfica de forma segura.
# ============================================================

import numpy as np

# ------------------------------------------------------------
# 1) Padronização das chaves
# ------------------------------------------------------------
# Boa prática:
# - Padronizar nomes de identificadores antes de realizar joins
# - Evitar ambiguidades e inconsistências futuras

df_conc = df_conc.rename(columns={"codigo": "codigo_concorrente"})
df_bairros = df_bairros.rename(columns={"codigo": "codigo_bairro"})
df_pop = df_pop.rename(columns={"codigo": "codigo_bairro"})


# ------------------------------------------------------------
# 2) Garantia de tipagem das chaves e métricas
# ------------------------------------------------------------
# Boa prática:
# - Garantir que chaves possuam o mesmo tipo
# - Utilizar Int64 (nullable) quando necessário
# - Garantir tipo numérico correto para cálculos

df_conc["codigo_bairro"] = df_conc["codigo_bairro"].astype("Int64")
df_bairros["codigo_bairro"] = df_bairros["codigo_bairro"].astype("Int64")
df_pop["codigo_bairro"] = df_pop["codigo_bairro"].astype("Int64")

df_pop["populacao"] = df_pop["populacao"].astype("Int64")
df_bairros["area"] = df_bairros["area"].astype("float64")


# ------------------------------------------------------------
# 3) Merge concorrentes + bairros
# ------------------------------------------------------------
# how="left":
# - Mantém todos os concorrentes
#
# validate="m:1":
# - Garante que cada bairro apareça no máximo uma vez na tabela de bairros
# - Protege contra duplicidade inesperada

geo = df_conc.merge(
    df_bairros,
    on="codigo_bairro",
    how="left",
    validate="m:1",
    suffixes=("_conc", "_bairro")
)


# ------------------------------------------------------------
# 4) Merge com população
# ------------------------------------------------------------
# validate="m:1" novamente garante integridade da dimensão

geo = geo.merge(
    df_pop,
    on="codigo_bairro",
    how="left",
    validate="m:1"
)


# ------------------------------------------------------------
# 5) Cálculo seguro da densidade demográfica
# ------------------------------------------------------------
# Boa prática:
# - Evitar divisão por zero
# - Utilizar np.where para cálculo vetorizado
# - Retornar NaN quando área inválida

geo["densidade_demografica"] = np.where(
    geo["area"] > 0,
    geo["populacao"] / geo["area"],
    np.nan
)


# ------------------------------------------------------------
# 6) Seleção explícita de colunas finais
# ------------------------------------------------------------
# Boa prática:
# - Selecionar apenas colunas necessárias
# - Renomear para padronização e clareza
# - Evitar carregar colunas auxiliares desnecessárias

concorrente_geografia = geo[[
    "codigo_concorrente",
    "nome_conc",
    "categoria",
    "faixa_preco",
    "codigo_bairro",
    "municipio_bairro",
    "uf_bairro",
    "area",
    "populacao",
    "densidade_demografica"
]].rename(columns={
    "nome_conc": "nome_concorrente",
    "municipio_bairro": "municipio",
    "uf_bairro": "uf"
}).copy()


# ------------------------------------------------------------
# 7) Ordenação final
# ------------------------------------------------------------
# Facilita leitura e inspeção

concorrente_geografia = concorrente_geografia.sort_values("codigo_concorrente")


# ------------------------------------------------------------
# 8) Conferência final
# ------------------------------------------------------------
# Boa prática:
# - Validar estrutura após integração
# - Confirmar colunas e volume final

print("Shape final:", concorrente_geografia.shape)

print("\nColunas finais:")
print(concorrente_geografia.columns)

concorrente_geografia.head()

Shape final: (4202, 10)

Colunas finais:
Index(['codigo_concorrente', 'nome_concorrente', 'categoria', 'faixa_preco',
       'codigo_bairro', 'municipio', 'uf', 'area', 'populacao',
       'densidade_demografica'],
      dtype='object')


,codigo_concorrente,nome_concorrente,categoria,faixa_preco,codigo_bairro,municipio,uf,area,populacao,densidade_demografica
2989,100685983380612,A Eli quem faz,Candy Store,3,3552403001,Sumaré,SP,13.3290,51200,3841.248406
3561,101025009987066,Café e Arte,"Restaurant, Coffee Shop",2,35095078,Campinas,SP,38.3206,38740,1010.944505
1980,101037933375826,Dona Formiga,Dessert Shop,0,<NA>,NaN,NaN,NaN,<NA>,NaN
886,101252426691498,Lanchonete Família Gianotti,Restaurant,2,<NA>,NaN,NaN,NaN,<NA>,NaN
502,101257903326467,Extra Itu,"Supermarket, Shopping & Retail",4,<NA>,NaN,NaN,NaN,<NA>,NaN


ENRIQUECER FLUXO COM DADOS DO CONCORRENTE

In [23]:
# ============================================================
# Integração final - Métricas de fluxo + Dimensão de concorrente
# Objetivo:
#   Enriquecer a tabela analítica de fluxo com informações
#   descritivas do concorrente (nome, categoria, faixa_preco).
# ============================================================

# ------------------------------------------------------------
# 1) Merge entre fato analítica e dimensão
# ------------------------------------------------------------
# fluxo_analitico: tabela fato agregada
# faixa_preco: dimensão descritiva do concorrente
#
# how="left":
# - Mantém todas as métricas calculadas
# - Evita perder registros analíticos caso dimensão esteja incompleta
#
# Boa prática:
# - Garantir que a chave esteja padronizada
# - Validar se não há duplicidade na dimensão antes do merge

fluxo_final = fluxo_analitico.merge(
    faixa_preco,
    on="codigo_concorrente",
    how="left"
)


# ------------------------------------------------------------
# 2) Seleção explícita de colunas finais
# ------------------------------------------------------------
# Boa prática:
# - Definir ordem lógica das colunas
# - Separar identificadores, atributos e métricas
# - Evitar carregar colunas auxiliares desnecessárias

fluxo_final = fluxo_final[[
    "codigo_concorrente",  # chave
    "nome",                # atributo descritivo
    "categoria",           # atributo categórico
    "faixa_preco",         # atributo econômico
    "dia_semana",          # dimensão temporal
    "periodo",             # dimensão temporal
    "media_fluxo",         # métrica
    "max_fluxo",           # métrica
    "min_fluxo",           # métrica
    "total_eventos"        # métrica
]]


# ------------------------------------------------------------
# 3) Resultado final
# ------------------------------------------------------------
# Tabela pronta para:
# - Análises comparativas
# - Dashboard BI
# - Exportação para camada refined/gold

fluxo_final

,codigo_concorrente,nome,categoria,faixa_preco,dia_semana,periodo,media_fluxo,max_fluxo,min_fluxo,total_eventos
0,101025009987066,Café e Arte,"Restaurant, Coffee Shop",2,Friday,manha,4.000000,6,3,16
1,101025009987066,Café e Arte,"Restaurant, Coffee Shop",2,Friday,noite,2.200000,3,1,11
2,101025009987066,Café e Arte,"Restaurant, Coffee Shop",2,Friday,tarde,5.250000,9,3,21
3,101025009987066,Café e Arte,"Restaurant, Coffee Shop",2,Monday,manha,3.600000,5,2,18
4,101025009987066,Café e Arte,"Restaurant, Coffee Shop",2,Monday,noite,2.600000,4,1,13
...,...,...,...,...,...,...,...,...,...,...
12267,1897386380476876,Espetinhos Do Luiz Franzoni,"Bar & Grill, Beer Garden, Product/Service",1,Thursday,tarde,1.333333,2,1,4
12268,1897386380476876,Espetinhos Do Luiz Franzoni,"Bar & Grill, Beer Garden, Product/Service",1,Tuesday,noite,1.000000,1,1,1
12269,1897386380476876,Espetinhos Do Luiz Franzoni,"Bar & Grill, Beer Garden, Product/Service",1,Tuesday,tarde,1.000000,1,1,3
12270,1897386380476876,Espetinhos Do Luiz Franzoni,"Bar & Grill, Beer Garden, Product/Service",1,Wednesday,noite,1.000000,1,1,2


Código atualizado para salvar em refined_gold

In [24]:
# ============================================================
# Persistência da camada Refined/Gold
# Objetivo:
#   Salvar as tabelas finais analíticas em formato Parquet,
#   prontas para consumo por BI, dashboards ou modelagem.
# ============================================================

from pathlib import Path


# ------------------------------------------------------------
# 1) Definição do caminho da camada Refined/Gold
# ------------------------------------------------------------
# Boa prática:
# - Separar claramente as camadas do pipeline:
#   raw → trusted → refined (gold)
# - Centralizar caminho base
# - Evitar caminhos absolutos fixos

OUTPUT_PATH = Path("../data/refined_gold")


# ------------------------------------------------------------
# 2) Garantir existência do diretório
# ------------------------------------------------------------
# parents=True:
# - Cria toda a estrutura caso não exista
#
# exist_ok=True:
# - Evita erro se a pasta já estiver criada

OUTPUT_PATH.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# 3) Persistência das tabelas finais
# ------------------------------------------------------------
# Uso de Parquet:
# - Formato colunar eficiente
# - Melhor compressão
# - Preserva tipagem (datetime, Int64, etc.)
# - Ideal para consumo analítico
#
# index=False:
# - Evita salvar índice do pandas desnecessariamente

fluxo_final.to_parquet(
    OUTPUT_PATH / "fluxo_concorrente_analitico.parquet",
    index=False
)

concorrente_geografia.to_parquet(
    OUTPUT_PATH / "concorrente_geografia.parquet",
    index=False
)


# ------------------------------------------------------------
# 4) Confirmação de salvamento
# ------------------------------------------------------------
# resolve() exibe caminho absoluto final
# Boa prática para rastreabilidade

print("Arquivos salvos com sucesso em:", OUTPUT_PATH.resolve())

Arquivos salvos com sucesso em: /home/kayo/projeto-geofusion/geofusion-case/data/refined_gold


Validar leitura da refined_gold

In [25]:
# ============================================================
# Validação da camada Refined/Gold
# Objetivo:
#   Ler as tabelas finais persistidas e validar se:
#   - Foram salvas corretamente
#   - Mantiveram tipagem esperada
#   - Estão prontas para consumo analítico
# ============================================================

from pathlib import Path
import pandas as pd


# ------------------------------------------------------------
# 1) Definição do caminho da camada Gold
# ------------------------------------------------------------
# Boa prática:
# - Manter padronização de nomenclatura entre camadas
# - Centralizar caminho base

GOLD_PATH = Path("../data/refined_gold")


# ------------------------------------------------------------
# 2) Leitura dos arquivos Parquet
# ------------------------------------------------------------
# Boa prática:
# - Validar leitura após persistência
# - Confirmar preservação de schema

fluxo_gold = pd.read_parquet(GOLD_PATH / "fluxo_concorrente_analitico.parquet")
geo_gold = pd.read_parquet(GOLD_PATH / "concorrente_geografia.parquet")


# ------------------------------------------------------------
# 3) Inspeção inicial dos dados
# ------------------------------------------------------------
# Exibe os 5 primeiros registros para validação visual.
# Em ambiente produtivo, recomenda-se também validar:
# - shape
# - dtypes
# - consistência de chaves

print("===== FLUXO_CONCORRENTE_ANALITICO =====")
display(fluxo_gold.head())

print("\n===== CONCORRENTE_GEOGRAFIA =====")
display(geo_gold.head())

===== FLUXO_CONCORRENTE_ANALITICO =====


,codigo_concorrente,nome,categoria,faixa_preco,dia_semana,periodo,media_fluxo,max_fluxo,min_fluxo,total_eventos
0,101025009987066,Café e Arte,"Restaurant, Coffee Shop",2,Friday,manha,4.00,6,3,16
1,101025009987066,Café e Arte,"Restaurant, Coffee Shop",2,Friday,noite,2.20,3,1,11
2,101025009987066,Café e Arte,"Restaurant, Coffee Shop",2,Friday,tarde,5.25,9,3,21
3,101025009987066,Café e Arte,"Restaurant, Coffee Shop",2,Monday,manha,3.60,5,2,18
4,101025009987066,Café e Arte,"Restaurant, Coffee Shop",2,Monday,noite,2.60,4,1,13



===== CONCORRENTE_GEOGRAFIA =====


,codigo_concorrente,nome_concorrente,categoria,faixa_preco,codigo_bairro,municipio,uf,area,populacao,densidade_demografica
0,100685983380612,A Eli quem faz,Candy Store,3,3552403001,Sumaré,SP,13.3290,51200,3841.248406
1,101025009987066,Café e Arte,"Restaurant, Coffee Shop",2,35095078,Campinas,SP,38.3206,38740,1010.944505
2,101037933375826,Dona Formiga,Dessert Shop,0,<NA>,None,None,NaN,<NA>,NaN
3,101252426691498,Lanchonete Família Gianotti,Restaurant,2,<NA>,None,None,NaN,<NA>,NaN
4,101257903326467,Extra Itu,"Supermarket, Shopping & Retail",4,<NA>,None,None,NaN,<NA>,NaN


Query para para os analistas extrairem as informações

In [26]:
# ============================================================
# Virtualização da camada Gold com DuckDB
# Objetivo:
#   Criar views diretamente a partir de arquivos Parquet,
#   permitindo consultas SQL sem necessidade de carregamento
#   prévio em memória.
# ============================================================

import duckdb
from pathlib import Path


# ------------------------------------------------------------
# 1) Definição do caminho da camada Gold
# ------------------------------------------------------------
# Boa prática:
# - Centralizar caminho base
# - Garantir consistência com pipeline anterior

GOLD_PATH = Path("../data/refined_gold")


# ------------------------------------------------------------
# 2) Criação da conexão DuckDB
# ------------------------------------------------------------
# Boa prática:
# - Conexão local in-memory (padrão)
# - Ideal para exploração analítica
# - Pode ser persistida em arquivo .duckdb se necessário

con = duckdb.connect()


# ------------------------------------------------------------
# 3) Criação de views a partir dos arquivos Parquet
# ------------------------------------------------------------
# read_parquet():
# - Lê diretamente do arquivo
# - Não carrega tudo em memória
# - Permite consultas SQL performáticas
#
# CREATE OR REPLACE VIEW:
# - Atualiza view caso já exista
# - Evita erro em reexecução do script
#
# Boa prática:
# - Utilizar views para virtualização
# - Separar camada física (Parquet) da lógica analítica (SQL)

con.execute(f"""
CREATE OR REPLACE VIEW fluxo_concorrente_analitico AS
SELECT *
FROM read_parquet('{GOLD_PATH / "fluxo_concorrente_analitico.parquet"}')
""")

con.execute(f"""
CREATE OR REPLACE VIEW concorrente_geografia AS
SELECT *
FROM read_parquet('{GOLD_PATH / "concorrente_geografia.parquet"}')
""")



In [27]:
# ============================================================
# Consulta analítica - Média geral de fluxo por dia e período
# Objetivo:
#   Calcular a média geral do fluxo considerando:
#       - dia_semana
#       - periodo
#   A partir da view fluxo_concorrente_analitico.
# ============================================================

# ------------------------------------------------------------
# 1) Definição da query SQL
# ------------------------------------------------------------
# AVG(media_fluxo):
# - Calcula média geral entre concorrentes
#
# GROUP BY:
# - Define granularidade da agregação
#
# ORDER BY:
# - Ordena por dia da semana (atenção: pode não respeitar ordem cronológica)

fluxo_concorrente_analitico = """
SELECT 
    dia_semana,
    periodo,
    AVG(media_fluxo) AS media_geral
FROM fluxo_concorrente_analitico
GROUP BY dia_semana, periodo
ORDER BY dia_semana;
"""


# ------------------------------------------------------------
# 2) Execução da consulta
# ------------------------------------------------------------
# .execute().df():
# - Executa SQL no DuckDB
# - Retorna resultado como DataFrame pandas
# - Permite integração imediata com Python

resultado1 = con.execute(fluxo_concorrente_analitico).df()


# ------------------------------------------------------------
# 3) Visualização inicial
# ------------------------------------------------------------
# head() exibe primeiras linhas para validação rápida

resultado1.head()

,dia_semana,periodo,media_geral
0,Friday,tarde,4.854306
1,Friday,manha,3.639118
2,Friday,noite,5.119109
3,Monday,tarde,4.563923
4,Monday,manha,3.656947


In [28]:
# ============================================================
# Consulta analítica - Top 5 por densidade demográfica
# Objetivo:
#   Identificar os concorrentes localizados em regiões com
#   maior densidade demográfica.
# ============================================================

# ------------------------------------------------------------
# 1) Definição da query SQL
# ------------------------------------------------------------
# SELECT:
# - Seleciona apenas colunas relevantes para análise
#
# ORDER BY densidade_demografica DESC:
# - Ordena da maior para a menor densidade
#
# LIMIT 5:
# - Retorna apenas os 5 primeiros registros (ranking)

concorrente_geografia = """
SELECT
    nome_concorrente,
    municipio,
    densidade_demografica
FROM concorrente_geografia
ORDER BY densidade_demografica DESC
LIMIT 5;
"""


# ------------------------------------------------------------
# 2) Execução da consulta
# ------------------------------------------------------------
# .execute().df():
# - Executa SQL no DuckDB
# - Converte o resultado para DataFrame pandas
# - Facilita visualização e análises posteriores

resultado2 = con.execute(concorrente_geografia).df()


# ------------------------------------------------------------
# 3) Resultado
# ------------------------------------------------------------
# Dataset pronto para:
# - Visualização
# - Cruzamento com fluxo
# - Análise estratégica de localização

resultado2

,nome_concorrente,municipio,densidade_demografica
0,Pão de Açúcar - Cambuí,Campinas,10757.789237
1,Mercado Municipal De Campinas,Campinas,10757.789237
2,Giovannetti Campinas,Campinas,10757.789237
3,Pink Elephant Campinas,Campinas,10757.789237
4,Deck 21,Campinas,10757.789237


Consultar sem criar view

In [29]:
# ============================================================
# Leitura direta de Parquet via DuckDB
# Objetivo:
#   Executar consulta SQL diretamente sobre arquivo Parquet
#   sem depender de view previamente criada.
# ============================================================

# ------------------------------------------------------------
# 1) Definição da query
# ------------------------------------------------------------
# read_parquet():
# - Permite consultar arquivo Parquet diretamente
# - Evita necessidade de registrar view previamente
#
# LIMIT 5:
# - Boa prática para testes rápidos
# - Evita carregar grande volume desnecessariamente

query = f"""
SELECT *
FROM read_parquet('{GOLD_PATH / "fluxo_concorrente_analitico.parquet"}')
LIMIT 5;
"""


# ------------------------------------------------------------
# 2) Execução da query
# ------------------------------------------------------------
# .execute().df():
# - Executa a consulta no DuckDB
# - Retorna o resultado como DataFrame pandas
# - Ideal para exploração interativa

con.execute(query).df()

,codigo_concorrente,nome,categoria,faixa_preco,dia_semana,periodo,media_fluxo,max_fluxo,min_fluxo,total_eventos
0,101025009987066,Café e Arte,"Restaurant, Coffee Shop",2,Friday,manha,4.00,6,3,16
1,101025009987066,Café e Arte,"Restaurant, Coffee Shop",2,Friday,noite,2.20,3,1,11
2,101025009987066,Café e Arte,"Restaurant, Coffee Shop",2,Friday,tarde,5.25,9,3,21
3,101025009987066,Café e Arte,"Restaurant, Coffee Shop",2,Monday,manha,3.60,5,2,18
4,101025009987066,Café e Arte,"Restaurant, Coffee Shop",2,Monday,noite,2.60,4,1,13
